<div align="center">

# <span style="color:#2E86C1;"> Example 1: Anscombe's Quartet</span>

**Author:** [<span style="color:#8E44AD;">Dr. D Bhanu Prakash</span>](https://dbhanuprakash233.github.io)

<img src="https://github.com/dbhanuprakash233/SSSIHL_DBP/blob/main/assets/SssihlLogo.png?raw=true" alt="University Logo" width="80"/>

**<span style="color:#16A085;">Sri Sathya Sai Institute of Higher Learning</span>**  
<span style="color:#5D6D7E;">Prasanthi Nilayam - 515 134, Andhra Pradesh, India.</span>

**Course:** <span style="color:#D35400;">Data Analysis and Visualisation</span>  
**Course Code:** <span style="color:#1ABC9C;">Minor</span>

</div>

In [ ]:
"""
Anscombe's Quartet - Statistical Simulation & Visualisation
=============================================================
Reproduces F.J. Anscombe's 1973 dataset ("Graphs in Statistical Analysis",
The American Statistician, 27(1), 17-21) and verifies numerically that all
four datasets share (almost) identical summary statistics, then plots them
to reveal how different they actually are.
"""

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1. The four datasets (n = 11 points each)
# ------------------------------------------------------------------
data = {
    "I":   {"x": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
             "y": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]},
    "II":  {"x": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
             "y": [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]},
    "III": {"x": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
             "y": [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]},
    "IV":  {"x": [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
             "y": [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]},
}

# ------------------------------------------------------------------
# 2. Compute summary statistics for each dataset
#    mean, sample variance (ddof=1), Pearson r, OLS regression, R^2
# ------------------------------------------------------------------
summary_rows = []
fitted = {}

for name, d in data.items():
    x = np.array(d["x"], dtype=float)
    y = np.array(d["y"], dtype=float)

    n = len(x)
    mean_x, mean_y = x.mean(), y.mean()
    var_x, var_y = x.var(ddof=1), y.var(ddof=1)          # sample variance
    cov_xy = np.cov(x, y, ddof=1)[0, 1]                   # sample covariance

    result = stats.linregress(x, y)                       # slope, intercept, r, p, se
    slope, intercept, r, p_value, std_err = result
    r_squared = r ** 2

    fitted[name] = (x, y, slope, intercept, r)

    summary_rows.append({
        "Dataset": name,
        "n": n,
        "mean(x)": round(mean_x, 2),
        "mean(y)": round(mean_y, 2),
        "var(x)": round(var_x, 2),
        "var(y)": round(var_y, 2),
        "cov(x,y)": round(cov_xy, 3),
        "r (Pearson)": round(r, 3),
        "R^2": round(r_squared, 3),
        "slope (b1)": round(slope, 3),
        "intercept (b0)": round(intercept, 3),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Dataset")
pd.set_option("display.width", 120)
print("=" * 70)
print("Summary statistics — notice how similar every column is:")
print("=" * 70)
print(summary_df)
print()

# ------------------------------------------------------------------
# 3. The maths behind those numbers (for reference / lecture notes)
#
#   mean:        x_bar = (1/n) * sum(x_i)
#   variance:    s_x^2 = (1/(n-1)) * sum((x_i - x_bar)^2)
#   covariance:  s_xy  = (1/(n-1)) * sum((x_i - x_bar)(y_i - y_bar))
#   correlation: r     = s_xy / (s_x * s_y)
#   OLS slope:   b1    = s_xy / s_x^2   =  r * (s_y / s_x)
#   OLS intercpt:b0    = y_bar - b1 * x_bar
#   R^2 (simple linear regression) = r^2
#
#   All four datasets were constructed by Anscombe so that x_bar, s_x^2,
#   y_bar, s_y^2, s_xy, r, b0, b1 and R^2 match to 2-3 decimal places,
#   even though the (x, y) relationships are structurally very different.
# ------------------------------------------------------------------

# ------------------------------------------------------------------
# 4. Visualise: 2x2 grid, one scatter + fitted line per dataset
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
titles = {
    "I":   "I: roughly linear, normal scatter",
    "II":  "II: clear curvature — linear fit is WRONG model",
    "III": "III: perfect line + one outlier distorts slope",
    "IV":  "IV: one high-leverage point creates the whole correlation",
}

for ax, (name, (x, y, slope, intercept, r)) in zip(axes.flat, fitted.items()):
    ax.scatter(x, y, color="crimson", s=50, zorder=3, edgecolor="black")
    xs = np.linspace(x.min() - 1.5, x.max() + 1.5, 50)
    ax.plot(xs, intercept + slope * xs, color="steelblue", lw=2, zorder=2)
    ax.set_title(f"Dataset {name}\nr = {r:.3f},  y = {intercept:.2f} + {slope:.2f}x",
                 fontsize=10)
    ax.set_xlim(2, 20)
    ax.set_ylim(2, 14)
    ax.grid(alpha=0.3)

fig.suptitle("Anscombe's Quartet: Identical Statistics, Different Stories",
             fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("anscombe_quartet.png", dpi=150)
print("Saved plot -> anscombe_quartet.png")
plt.show()

# ------------------------------------------------------------------
# 5. OPTIONAL — bonus simulation:
#    Confirm how fragile "matching stats" really is by perturbing one
#    point in Dataset I and watching r / slope drift, to contrast with
#    how Dataset III's stats depend entirely on ONE outlier.
# ------------------------------------------------------------------
print("\nBonus: effect of removing the outlier in Dataset III")
x3, y3 = np.array(data["III"]["x"]), np.array(data["III"]["y"])
outlier_idx = np.argmax(y3)  # point (13, 12.74)
mask = np.ones(len(x3), dtype=bool)
mask[outlier_idx] = False

r_with = stats.linregress(x3, y3).rvalue
r_without = stats.linregress(x3[mask], y3[mask]).rvalue
print(f"  r WITH outlier    : {r_with:.3f}")
print(f"  r WITHOUT outlier : {r_without:.3f}  (should jump to ~1.00 — near perfect line!)")